<a href="https://colab.research.google.com/github/aclaire10/Datasci-112-Final-Project/blob/main/Datasci_112_Final_Project_Merging_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Merging Data for Historic DC Dataset

This was the code that we used to merge all of our original datasets into one large dataset that contained all of the features we cared about. This was the foundadtion of our project and served as our historical model.[link text](https://)

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, Polygon
import pyproj
import fiona
import pyogrio
import openpyxl

In [ ]:
#!/usr/bin/env python3
"""
build_final_cleaned_with_business_tax_proxy.py

One-command pipeline to reproduce:
  ~/Downloads/datasci112_final_cleaned_with_business_tax_proxy.csv

This script stitches together the two existing project steps:
  1) build_master_dataset.py  -> outputs/master_county_dataset.csv
  2) add_business_tax_proxy.py -> adds gross receipts/business tax proxy columns

It writes an intermediate file expected by add_business_tax_proxy.py:
  ~/Downloads/datasci112_final_cleaned.csv

Then it writes the final file:
  ~/Downloads/datasci112_final_cleaned_with_business_tax_proxy.csv

Inputs
------
Same as build_master_dataset.py (either in ./data/ or ~/Downloads):
  - counties_data.zip
  - data_center.geojson
  - co-est2024-alldata.csv
  - monthly_industrial_electricity_prices.xlsx
  - generation_monthly_2024.csv  (preferred; build_master_dataset.py uses this)
Optional:
  - state corporate tax file (see build_master_dataset.py PATH_TAX_CSV)

Usage
-----
  python build_final_cleaned_with_business_tax_proxy.py

If your inputs live elsewhere, set:
  export DATA_DIR=/path/to/data
and re-run.
"""

from __future__ import annotations

import shutil
from pathlib import Path


def main() -> int:
    base_dir = Path(__file__).resolve().parent
    downloads = Path.home() / "Downloads"

    # Step 1: build master dataset (writes to ./outputs/)
    import build_master_dataset as bmd

    bmd.main()

    master_csv = base_dir / "outputs" / "master_county_dataset.csv"
    if not master_csv.exists():
        raise FileNotFoundError(f"Expected output not found: {master_csv}")

    # Step 2: write the intermediate file expected by add_business_tax_proxy.py
    intermediate = downloads / "datasci112_final_cleaned.csv"
    shutil.copyfile(master_csv, intermediate)

    # Step 3: add business tax proxy (writes to ~/Downloads/)
    import add_business_tax_proxy as tax

    tax.main()

    final_out = downloads / "datasci112_final_cleaned_with_business_tax_proxy.csv"
    if not final_out.exists():
        raise FileNotFoundError(f"Expected final output not found: {final_out}")

    print(f"Saved: {final_out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())